In [ ]:
# KALMAN DERIVED REBUILD v2.2 — LEDGER CONTRACT / ONE CELL
# Research-only. Uses original FIXED_4 ledger as the exact gross/net contract.
from google.colab import drive
drive.mount("/content/drive",force_remount=False)
from pathlib import Path
from datetime import datetime,timezone
import json,shutil
import numpy as np,pandas as pd

R=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results")
ROOT=R/"open_revalidation_v1"
AUD=ROOT/"open_revalidation_trade_audit.parquet"
LED=R/"exit_policy_v1_0_pre2026/exit_policy_v1_0_1_trade_ledger.parquet"
a=pd.read_parquet(AUD); l=pd.read_parquet(LED)
l=l[l["policy"].eq("FIXED_4")].copy()
keys=["policy","fold","entry_timestamp","exit_timestamp","entry_seq","exit_seq","symbol"]
for c in ["entry_timestamp","exit_timestamp"]:
 a[c]=pd.to_datetime(a[c],utc=True); l[c]=pd.to_datetime(l[c],utc=True)

# Exact 1:1 ledger identity is mandatory.
if l.duplicated(keys).any(): raise RuntimeError("FIXED_4 ledger keys are not unique.")
m=a.merge(l[keys+["gross_return","net_return","weight","holding_bars","exit_reason"]],on=keys,how="left",suffixes=("","__ledger"),validate="one_to_one",indicator=True)
matched=m["_merge"].eq("both")
print("[LEDGER MATCH]",int(matched.sum()),"/",len(m))
if int(matched.sum())!=len(a): 
 print(m.loc[~matched,keys].to_string(index=False))
 raise RuntimeError("Audit does not map 1:1 to historical FIXED_4 ledger; stop.")

# Verify current audit contract against ledger before changing anything.
for c in ["gross_return","net_return","weight"]:
 lc=c+"__ledger"
 err=np.nanmax(np.abs(pd.to_numeric(m[c],errors="coerce")-pd.to_numeric(m[lc],errors="coerce")))
 print("[VERIFY]",c,"max_abs_err=",err)
 if not np.isfinite(err) or err>1e-12: raise RuntimeError(f"Audit {c} differs from original ledger.")

# Recompute only price-derived diagnostics for IEX-complete rows.
price=["entry_price_iex","fixed4_exit_price_iex","prev_close_price_iex","open_0_price_iex","open_5_price_iex","open_15_price_iex"]
for c in price: m[c]=pd.to_numeric(m[c],errors="coerce")
complete=m[price].notna().all(axis=1)
def lr(exitp,entryp): return exitp/entryp-1
m.loc[complete,"position_return_prev_close"]=lr(m.loc[complete,"prev_close_price_iex"],m.loc[complete,"entry_price_iex"])
m.loc[complete,"position_return_open"]=lr(m.loc[complete,"open_0_price_iex"],m.loc[complete,"entry_price_iex"])
m.loc[complete,"position_return_5m"]=lr(m.loc[complete,"open_5_price_iex"],m.loc[complete,"entry_price_iex"])
m.loc[complete,"position_return_15m"]=lr(m.loc[complete,"open_15_price_iex"],m.loc[complete,"entry_price_iex"])
m.loc[complete,"overnight_gap_return"]=m.loc[complete,"open_0_price_iex"]/m.loc[complete,"prev_close_price_iex"]-1
m.loc[complete,"open_momentum_5m"]=m.loc[complete,"open_5_price_iex"]/m.loc[complete,"open_0_price_iex"]-1
m.loc[complete,"open_momentum_15m"]=m.loc[complete,"open_15_price_iex"]/m.loc[complete,"open_0_price_iex"]-1
m.loc[complete,"giveback_prev_close_to_5m"]=m.loc[complete,"position_return_prev_close"]-m.loc[complete,"position_return_5m"]
m.loc[complete,"reconstructed_fixed4_raw_return"]=lr(m.loc[complete,"fixed4_exit_price_iex"],m.loc[complete,"entry_price_iex"])

# Critical correction:
# original strategy net return = original ledger net_return.
# Do not derive it from IEX raw price return. IEX is an execution/revalidation measurement layer.
m.loc[complete,"reconstructed_fixed4_net_return"]=m.loc[complete,"net_return__ledger"]

# Keep original 61-column schema only.
orig_cols=list(a.columns)
out=m[orig_cols].copy()
derived=["position_return_prev_close","position_return_open","position_return_5m","position_return_15m","overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m","reconstructed_fixed4_raw_return","reconstructed_fixed4_net_return"]
print("\n[COMPLETENESS]")
print(out[derived].notna().sum().to_string())

# Existing 20 rows must preserve reconstructed net exactly.
old=a["reconstructed_fixed4_net_return"].notna()
preserve=np.nanmax(np.abs(pd.to_numeric(out.loc[old,"reconstructed_fixed4_net_return"])-pd.to_numeric(a.loc[old,"reconstructed_fixed4_net_return"])))
print("[LEGACY NET PRESERVATION] max_abs_err=",preserve)
if not np.isfinite(preserve) or preserve>1e-12: raise RuntimeError("Legacy reconstructed net changed; stop.")

before=int(a["reconstructed_fixed4_net_return"].notna().sum()); after=int(out["reconstructed_fixed4_net_return"].notna().sum())
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cand=ROOT/"open_revalidation_trade_audit.derived_v2_2_candidate.parquet"
bak=ROOT/f"open_revalidation_trade_audit.pre_derived_v2_2_{stamp}.parquet"
out.to_parquet(cand,index=False)
if after>before:
 shutil.copy2(AUD,bak); cand.replace(AUD); action="REPLACED"
else: action="CANDIDATE_ONLY"
rep={"schema":"kalman-open-revalidation-derived-v2.2","research_only":True,"production_changed":False,"live_trading":False,"neon_write":False,"rows":len(out),"ledger_match":int(matched.sum()),"price_complete":int(complete.sum()),"derived_before":before,"derived_after":after,"legacy_net_max_abs_err":float(preserve),"action":action,"backup":str(bak) if action=="REPLACED" else None}
(ROOT/"derived_rebuild_v2_2_report.json").write_text(json.dumps(rep,indent=2))
print("\n",json.dumps(rep,indent=2))
print("\nIMPORTANT: OPEN_* policy columns were preserved, not recomputed.")
print("NEXT: run a corrected validation that compares frozen policy returns to the ledger baseline; do not compare net_return to itself.")
